# Macaulay Equity Duration (Scaled Explicit Block + Price-Implied Terminal Value)


## 1) Load data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
from pathlib import Path

from plot_style import COLORS, set_global_plot_style, style_axes, style_legend, style_time_axis

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)

# Use project-wide static plot style (non-interactive notebook plots)
set_global_plot_style()

# Prefer local dataset filename; fallback to existing Project_Data layout
input_candidates = [
    Path('euro500_macaulay.parquet'),
    Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_macaulay.parquet'),
]
INPUT_PATH = next((p for p in input_candidates if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError('Could not find euro500_macaulay.parquet in expected locations.')

# Requested final output filenames
OUTPUT_PATH = Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/EQDuration_Macaulay.parquet')
STEP11_FIG_PATH = Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/graphs/EQDuration_Macaulay_step11_coverage.png')

# Full Euro500 base table (kept for final remapping in steps 11 and 13)
df_full = pd.read_parquet(INPUT_PATH).copy()
df_full['_row_id'] = np.arange(len(df_full))

# Working table for duration estimation (will be filtered)
df = df_full.copy()

print(f'Loaded rows (full Euro500 base): {len(df_full):,}')
print(f'Loaded columns: {len(df_full.columns):,}')
print(f'Input file: {INPUT_PATH.resolve()}')


In [ ]:
# Required fields for this model
required_cols = [
    'PriceClose',
    'EPS_FY1',
    'EPS_FY2',
    'EPS_FY3',
]

# Required data cleaning
# - drop missing PriceClose / EPS_FY1 / EPS_FY2 / EPS_FY3
# - drop PriceClose <= 0 and EPS_FY1 <= 0
# - do NOT drop based on EPS_FY4 / EPS_FY5 missingness
df = df.dropna(subset=required_cols).copy()
df = df[(df['PriceClose'] > 0) & (df['EPS_FY1'] > 0)].copy()

# Common safeguard flag used by all discount-rate versions
df['flag_small_price'] = df['PriceClose'] <= 1

print(f'Rows after required cleaning: {len(df):,}')


## 2) Model constants and discount-rate versions


In [ ]:
# Shared constants for CURRENT specification
H = 8
G_INF = 0.02
G0_LOWER = -0.02
G0_UPPER = 0.08
SCALE_FLOOR = 0.80

RATES = {
    'r10': 0.10,
    'r125': 0.125,
}

print(f'Explicit horizon H = {H}')
print(f'LTG clip range for g0: [{G0_LOWER:.2%}, {G0_UPPER:.2%}]')
print(f'Long-run anchor growth g_inf = {G_INF:.2%}')
print(f'Scaling corridor floor = {SCALE_FLOOR:.0%}')
print(f'Discount-rate versions: {RATES}')


## 3) LTG preparation


In [ ]:
# LTG is used as medium-run transition signal (not perpetuity growth)
# Convert percent to decimal once; clipping is spec-dependent (old vs new)
df['LTG_decimal'] = pd.to_numeric(df['LTG'], errors='coerce') / 100.0


## 4) Explicit cashflows (years 1-3)


In [ ]:
df['C1'] = pd.to_numeric(df['EPS_FY1'], errors='coerce')
df['C2'] = pd.to_numeric(df['EPS_FY2'], errors='coerce')
df['C3'] = pd.to_numeric(df['EPS_FY3'], errors='coerce')


## 5) Transition growth path (years 4-8)


In [ ]:
# Build CURRENT transition path (H=8, g0 clipped to [-2%, 8%])
# Growth fades linearly from year 4 to year 8
df['g0_raw'] = df['LTG_decimal']
df['g0'] = df['g0_raw'].clip(lower=G0_LOWER, upper=G0_UPPER).fillna(G_INF)

for t in range(4, H + 1):
    df[f'g_{t}'] = df['g0'] + ((t - 4) / (H - 4)) * (G_INF - df['g0'])

df['C4'] = df['C3'] * (1.0 + df['g_4'])
for t in range(5, H + 1):
    df[f'C{t}'] = df[f'C{t-1}'] * (1.0 + df[f'g_{t}'])


## 6) Capped Scaling Corridor + Price-Implied Terminal Block (No Gordon TV)


In [ ]:
price = df['PriceClose'].to_numpy(dtype=float)
base_c1 = df['C1'].to_numpy(dtype=float)
base_c2 = df['C2'].to_numpy(dtype=float)
base_c3 = df['C3'].to_numpy(dtype=float)
ltg_dec = df['LTG_decimal']

def build_cashflows(horizon: int, g0_lower: float, g0_upper: float):
    # Return cashflow matrix C1..CH and clipped g0 for current spec.
    n = len(df)
    cf = np.empty((n, horizon), dtype=float)
    cf[:, 0] = base_c1
    cf[:, 1] = base_c2
    cf[:, 2] = base_c3

    g0 = ltg_dec.clip(lower=g0_lower, upper=g0_upper).fillna(G_INF).to_numpy(dtype=float)
    prev = cf[:, 2].copy()

    for t in range(4, horizon + 1):
        g_t = g0 + ((t - 4) / (horizon - 4)) * (G_INF - g0)
        prev = prev * (1.0 + g_t)
        cf[:, t - 1] = prev

    return cf, g0

def run_current_spec(frame: pd.DataFrame, horizon: int, g0_lower: float, g0_upper: float, scale_floor: float):
    cf_mat, _ = build_cashflows(horizon=horizon, g0_lower=g0_lower, g0_upper=g0_upper)

    for suffix, r in RATES.items():
        disc = np.power(1.0 + r, np.arange(1, horizon + 1, dtype=float))

        # Step 1: unscaled explicit PV
        pv_raw_mat = cf_mat / disc
        pv_explicit_raw = pv_raw_mat.sum(axis=1)

        # Step 2: raw scaling factor
        sf_raw = np.divide(
            price,
            pv_explicit_raw,
            out=np.ones_like(price),
            where=pv_explicit_raw > 0,
        )

        # Step 3: capped scaling corridor
        scale_factor = np.where(
            pv_explicit_raw > 0,
            np.where(sf_raw >= 1.0, 1.0, np.maximum(scale_floor, sf_raw)),
            1.0,
        )

        # Step 4: scaled cashflows and final explicit PV
        cf_scaled = cf_mat * scale_factor[:, None]
        pv_mat = cf_scaled / disc
        pv_explicit = pv_mat.sum(axis=1)

        # Price-implied terminal block
        pv_terminal = price - pv_explicit

        # Macaulay duration with horizon-consistent terminal maturity
        t_vec = np.arange(1, horizon + 1, dtype=float)
        t_terminal = horizon + (1.0 + r) / r
        duration_num = pv_mat @ t_vec + t_terminal * pv_terminal
        duration = duration_num / price

        # Core diagnostics
        valuation_gap = np.divide(pv_explicit, price, out=np.full_like(price, np.nan), where=price > 0)

        # Cleaning flags
        flag_negative_terminal = pv_terminal < 0
        flag_bad_duration = (duration <= 0) | (duration > 100)
        invalid = flag_negative_terminal | frame['flag_small_price'].to_numpy() | flag_bad_duration

        duration_clean = duration.copy()
        duration_clean[invalid] = np.nan

        valid_clean = np.isfinite(duration_clean)
        if valid_clean.any():
            q01 = float(np.nanquantile(duration_clean, 0.01))
            q99 = float(np.nanquantile(duration_clean, 0.99))
            duration_clean = np.clip(duration_clean, q01, q99)
        else:
            q01 = np.nan
            q99 = np.nan

        frame[f'scale_factor_{suffix}'] = scale_factor
        frame[f'sf_raw_{suffix}'] = sf_raw
        frame[f'PV_explicit_raw_{suffix}'] = pv_explicit_raw
        frame[f'PV_explicit_{suffix}'] = pv_explicit
        frame[f'PV_terminal_{suffix}'] = pv_terminal
        frame[f'valuation_gap_{suffix}'] = valuation_gap
        frame[f'T_terminal_{suffix}'] = t_terminal
        frame[f'Duration_num_{suffix}'] = duration_num
        frame[f'duration_macaulay_{suffix}'] = duration
        frame[f'duration_macaulay_{suffix}_clean'] = duration_clean
        frame[f'flag_negative_terminal_{suffix}'] = flag_negative_terminal
        frame[f'flag_bad_duration_{suffix}'] = flag_bad_duration
        frame[f'flag_invalid_{suffix}'] = invalid
        frame[f'flag_scaled_{suffix}'] = scale_factor < 1.0
        frame[f'winsor_q01_{suffix}'] = q01
        frame[f'winsor_q99_{suffix}'] = q99

        print(f"[{suffix}] winsor bounds on cleaned duration: [{q01:.6f}, {q99:.6f}]")

run_current_spec(
    frame=df,
    horizon=H,
    g0_lower=G0_LOWER,
    g0_upper=G0_UPPER,
    scale_floor=SCALE_FLOOR,
)

# Backward-compatible aliases: main spec uses r=10%
df['duration_macaulay'] = df['duration_macaulay_r10']
df['duration_macaulay_clean'] = df['duration_macaulay_r10_clean']
df['PV_explicit'] = df['PV_explicit_r10']
df['PV_terminal'] = df['PV_terminal_r10']
df['valuation_gap'] = df['valuation_gap_r10']
df['flag_negative_terminal'] = df['flag_negative_terminal_r10']
df['flag_bad_duration'] = df['flag_bad_duration_r10']


## 7) Macaulay Duration with Price Anchor (H = 8)


In [ ]:
# Headline structural summary for CURRENT specification
for suffix, r in RATES.items():
    dur_col = f'duration_macaulay_{suffix}'
    clean_col = f'duration_macaulay_{suffix}_clean'
    neg_col = f'flag_negative_terminal_{suffix}'
    sf_col = f'scale_factor_{suffix}'

    print(
        f"{suffix} (r={r:.2%}) | mean raw duration: {df[dur_col].mean():.4f} | "
        f"mean clean duration: {df[clean_col].mean():.4f} | "
        f"share negative terminal: {df[neg_col].mean():.4%} | "
        f"share scaled: {(df[sf_col] < 1.0).mean():.4%}"
    )


## 8) Safeguards and Cleaning (by discount-rate version)


In [ ]:
# Keep generic clean alias for compatibility
df['duration_macaulay_clean'] = df['duration_macaulay_r10_clean']


## 9) Diagnostics (A-L for both rates + filtering transparency)


In [ ]:
def print_version_diagnostics(frame: pd.DataFrame, suffix: str, r: float) -> None:
    dur_col = f'duration_macaulay_{suffix}'
    clean_col = f'duration_macaulay_{suffix}_clean'
    neg_col = f'flag_negative_terminal_{suffix}'
    bad_col = f'flag_bad_duration_{suffix}'
    sf_col = f'scale_factor_{suffix}'
    vg_col = f'valuation_gap_{suffix}'

    print('=' * 90)
    print(f'Diagnostics for CURRENT spec {suffix} (r={r:.2%})')

    # A) raw duration describe()
    print()
    print('A) raw duration describe()')
    print(frame[dur_col].describe())

    # B) cleaned duration describe()
    print()
    print('B) cleaned duration describe()')
    print(frame[clean_col].describe())

    # C) quintile median cleaned durations
    valid = frame[clean_col].dropna()
    print()
    print('C) quintile median cleaned durations')
    if len(valid) >= 5:
        q = pd.qcut(valid, q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
        print(valid.groupby(q, observed=False).median())
    else:
        print('Insufficient non-missing observations for quintiles.')

    # D) cross-sectional std by date
    if 'date' in frame.columns:
        cs_std_date = frame.groupby('date', dropna=False)[clean_col].std()
        print()
        print('D) cross-sectional std by date (describe)')
        print(cs_std_date.describe())

    # E) share negative terminal residual
    print()
    print(f'E) share negative terminal residual: {frame[neg_col].mean():.4%}')

    # F) share retained after cleaning
    print(f'F) share retained after cleaning: {frame[clean_col].notna().mean():.4%}')

    # G) share scaled
    print(f'G) share scaled (scale_factor < 1): {(frame[sf_col] < 1.0).mean():.4%}')

    # H) scale_factor describe()
    print()
    print('H) scale_factor describe()')
    print(frame[sf_col].describe())

    # I) valuation_gap describe()
    print()
    print('I) valuation_gap describe()')
    print(frame[vg_col].describe())

    # J/K/L) valuation gap threshold shares
    print()
    print(f'J) share valuation_gap > 1.0: {(frame[vg_col] > 1.0).mean():.4%}')
    print(f'K) share valuation_gap > 1.1: {(frame[vg_col] > 1.1).mean():.4%}')
    print(f'L) share valuation_gap > 1.2: {(frame[vg_col] > 1.2).mean():.4%}')

    # Transparent filtering report
    small_share = frame['flag_small_price'].mean()
    neg_share = frame[neg_col].mean()
    bad_share = frame[bad_col].mean()
    total_retention = frame[clean_col].notna().mean()

    print()
    print('Filtering transparency:')
    print(f'- share removed by flag_small_price: {small_share:.4%}')
    print(f'- share removed by flag_negative_terminal: {neg_share:.4%}')
    print(f'- share removed by flag_bad_duration: {bad_share:.4%}')
    print(f'- total retention after cleaning: {total_retention:.4%}')

for suffix, r in RATES.items():
    print_version_diagnostics(df, suffix=suffix, r=r)


## 10) Visual Diagnostics

Die folgenden Plots machen die Stabilitätsdiagnose anschaulicher für beide Diskontsätze.


In [ ]:
# Compact visual summary and stability plots
summary_rows = []
for suffix, r in RATES.items():
    clean_col = f'duration_macaulay_{suffix}_clean'
    neg_col = f'flag_negative_terminal_{suffix}'
    sf_col = f'scale_factor_{suffix}'
    vg_col = f'valuation_gap_{suffix}'

    summary_rows.append({
        'spec': suffix,
        'r': r,
        'n_obs': int(len(df)),
        'mean_duration_clean': float(df[clean_col].mean()),
        'std_duration_clean': float(df[clean_col].std()),
        'share_negative_terminal': float(df[neg_col].mean()),
        'share_retained_clean': float(df[clean_col].notna().mean()),
        'share_scaled': float((df[sf_col] < 1.0).mean()),
        'share_vgap_gt_1': float((df[vg_col] > 1.0).mean()),
        'share_vgap_gt_1_1': float((df[vg_col] > 1.1).mean()),
        'share_vgap_gt_1_2': float((df[vg_col] > 1.2).mean()),
    })

summary_df = pd.DataFrame(summary_rows)
print('Summary table (structural stability):')
print(summary_df)

for suffix, r in RATES.items():
    raw_col = f'duration_macaulay_{suffix}'
    clean_col = f'duration_macaulay_{suffix}_clean'
    sf_col = f'scale_factor_{suffix}'
    vg_col = f'valuation_gap_{suffix}'

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(f'Visual Diagnostics: {suffix} (r={r:.2%})', fontsize=13)

    # 1) Duration distribution (raw vs cleaned)
    raw = df[raw_col].replace([np.inf, -np.inf], np.nan).dropna()
    clean = df[clean_col].replace([np.inf, -np.inf], np.nan).dropna()

    if len(raw) > 0:
        p_lo, p_hi = np.nanpercentile(raw, [1, 99])
        raw_clip = raw.clip(lower=p_lo, upper=p_hi)
        axes[0, 0].hist(raw_clip, bins=60, alpha=0.40, label='raw (1-99% clipped)', color=COLORS['blue_light'])
    if len(clean) > 0:
        axes[0, 0].hist(clean, bins=60, alpha=0.65, label='clean', color=COLORS['blue'])
    axes[0, 0].set_title('Duration Distribution')
    axes[0, 0].set_xlabel('Duration (years)')
    axes[0, 0].set_ylabel('Count')
    style_axes(axes[0, 0], grid_axis='y')
    style_legend(axes[0, 0], loc='upper right')

    # 2) Scale factor distribution
    sf = df[sf_col].replace([np.inf, -np.inf], np.nan).dropna()
    axes[0, 1].hist(sf, bins=40, color=COLORS['orange'], alpha=0.85)
    axes[0, 1].axvline(1.0, color=COLORS['reference'], linestyle='--', linewidth=1)
    axes[0, 1].axvline(SCALE_FLOOR, color=COLORS['red'], linestyle=':', linewidth=1)
    axes[0, 1].set_title('Scale Factor Distribution')
    axes[0, 1].set_xlabel('scale_factor')
    axes[0, 1].set_ylabel('Count')
    style_axes(axes[0, 1], grid_axis='y')

    # 3) Valuation gap distribution
    vg = df[vg_col].replace([np.inf, -np.inf], np.nan).dropna()
    vg_clip = vg.clip(lower=-0.5, upper=2.0)
    axes[1, 0].hist(vg_clip, bins=60, color=COLORS['green'], alpha=0.85)
    axes[1, 0].axvline(1.0, color=COLORS['reference'], linestyle='--', linewidth=1)
    axes[1, 0].axvline(1.1, color=COLORS['red'], linestyle=':', linewidth=1)
    axes[1, 0].axvline(1.2, color=COLORS['red'], linestyle=':', linewidth=1)
    axes[1, 0].set_title('Valuation Gap Distribution (clipped to [-0.5, 2.0])')
    axes[1, 0].set_xlabel('valuation_gap = PV_explicit / PriceClose')
    axes[1, 0].set_ylabel('Count')
    style_axes(axes[1, 0], grid_axis='y')

    # 4) Quintile medians
    valid = df[clean_col].dropna()
    if len(valid) >= 5:
        q = pd.qcut(valid, q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
        med = valid.groupby(q, observed=False).median()
        axes[1, 1].bar(med.index.astype(str), med.values, color=COLORS['primary'], alpha=0.9)
    axes[1, 1].set_title('Median Duration by Quintile (cleaned)')
    axes[1, 1].set_xlabel('Quintile')
    axes[1, 1].set_ylabel('Median duration')
    style_axes(axes[1, 1], grid_axis='y')

    plt.tight_layout()
    plt.show()

    # Time series panels (if date exists)
    if 'date' in df.columns:
        ts = df[['date', clean_col]].dropna().copy()
        ts['date'] = pd.to_datetime(ts['date'], errors='coerce')
        ts = ts.dropna(subset=['date'])

        if len(ts) > 0:
            g = ts.groupby('date')[clean_col]
            ts_stats = pd.DataFrame({
                'median': g.median(),
                'p25': g.quantile(0.25),
                'p75': g.quantile(0.75),
                'std': g.std(),
            }).dropna(how='all').sort_index()

            if len(ts_stats) > 0:
                fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
                fig.suptitle(f'Time-Series Diagnostics: {suffix} (r={r:.2%})', fontsize=13)

                ax[0].plot(ts_stats.index, ts_stats['median'], label='median', color=COLORS['blue'])
                ax[0].fill_between(ts_stats.index, ts_stats['p25'], ts_stats['p75'], alpha=0.25, color=COLORS['blue_light'], label='IQR')
                ax[0].set_ylabel('Duration')
                ax[0].set_title('Cross-sectional median + IQR (cleaned)')
                style_axes(ax[0], grid_axis='y')
                style_legend(ax[0], loc='upper left')

                ax[1].plot(ts_stats.index, ts_stats['std'], color=COLORS['purple'])
                ax[1].set_ylabel('Std')
                ax[1].set_title('Cross-sectional std by date (cleaned)')
                ax[1].set_xlabel('Date')
                style_axes(ax[1], grid_axis='y')
                style_time_axis(ax[1], x_min=ts_stats.index.min(), x_max=ts_stats.index.max(), x_ticks=ts_stats.index)

                plt.tight_layout()
                plt.show()


## 11) Coverage Analyse (Finale Duration nach Quartalen)

Coverage basiert auf `duration_macaulay_clean` (finale clean Duration der Hauptspezifikation).


In [ ]:
# Build full-Euro500 mapped output (final columns only) for steps 11 and 13
final_output_cols = [
    'duration_macaulay_r10',
    'duration_macaulay_r10_clean',
    'duration_macaulay_r125',
    'duration_macaulay_r125_clean',
    'PV_explicit_r10',
    'PV_explicit_r125',
    'PV_terminal_r10',
    'PV_terminal_r125',
    'valuation_gap_r10',
    'valuation_gap_r125',
    'scale_factor_r10',
    'scale_factor_r125',
    'flag_small_price',
    'flag_negative_terminal_r10',
    'flag_negative_terminal_r125',
    'flag_bad_duration_r10',
    'flag_bad_duration_r125',
    'flag_invalid_r10',
    'flag_invalid_r125',
    'flag_scaled_r10',
    'flag_scaled_r125',
]

missing_cols = [c for c in final_output_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required final output columns before remap: {missing_cols}')

# Keep only technical merge keys + final outputs in remapped frame
id_cols = [c for c in ['RIC', 'firm_id', 'effective_date', 'date'] if c in df_full.columns]

df_euro500_final = (
    df_full[['_row_id'] + id_cols]
    .merge(df[['_row_id'] + final_output_cols], on='_row_id', how='left')
)

print(f'Full Euro500 rows after remap: {len(df_euro500_final):,}')
print(f'Rows with final duration available: {df_euro500_final["duration_macaulay_r10_clean"].notna().sum():,}')

# Coverage analysis by quarter for final duration on FULL remapped Euro500
cov_duration_col = 'duration_macaulay_r10_clean'

if 'date' not in df_euro500_final.columns:
    print('Column `date` not found. Quarterly coverage analysis skipped.')
else:
    cov = df_euro500_final[['date', cov_duration_col]].copy()
    cov['date'] = pd.to_datetime(cov['date'], errors='coerce')
    cov = cov.dropna(subset=['date'])

    if len(cov) == 0:
        print('No valid dates for quarterly coverage analysis.')
    else:
        cov['quarter_period'] = cov['date'].dt.to_period('Q')

        q_cov = (
            cov.groupby('quarter_period', dropna=False)
            .agg(
                n_total=(cov_duration_col, 'size'),
                n_available=(cov_duration_col, lambda s: s.notna().sum()),
                median_duration=(cov_duration_col, 'median'),
                p25_duration=(cov_duration_col, lambda s: s.quantile(0.25)),
                p75_duration=(cov_duration_col, lambda s: s.quantile(0.75)),
            )
            .reset_index()
            .sort_values('quarter_period')
        )

        q_cov['quarter'] = q_cov['quarter_period'].astype(str)
        q_cov['quarter_ts'] = q_cov['quarter_period'].dt.to_timestamp()
        q_cov['coverage_share'] = q_cov['n_available'] / q_cov['n_total']

        print('Quarterly coverage table for final duration (FULL Euro500 remap):')
        print(q_cov[['quarter', 'n_total', 'n_available', 'median_duration', 'p25_duration', 'p75_duration', 'coverage_share']])

        fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

        ax[0].bar(q_cov['quarter_ts'], q_cov['coverage_share'], width=70, color=COLORS['blue_light'], alpha=0.9)
        ref_mask = q_cov['coverage_share'].ge(0) & q_cov['coverage_share'].notna()
        ref_median = q_cov.loc[ref_mask, 'coverage_share'].median()
        if pd.notna(ref_median):
            ax[0].axhline(ref_median, color=COLORS['orange'], linestyle='--', linewidth=1, label=f"Median non-negative quarters ({ref_median:.1%})")
        ax[0].set_ylim(0, 1.0)
        ax[0].set_ylabel('Coverage share (Euro 500)')
        ax[0].set_title('Coverage of Final Duration by Quarter')
        style_axes(ax[0], grid_axis='y')
        style_legend(ax[0], loc='lower right')

        ax[1].plot(q_cov['quarter_ts'], q_cov['median_duration'], color=COLORS['blue'], linewidth=0.9, marker='o', markersize=2.0, label='median')
        ax[1].fill_between(q_cov['quarter_ts'], q_cov['p25_duration'], q_cov['p75_duration'], color=COLORS['blue_light'], alpha=0.25, label='IQR')
        ax[1].set_ylabel('Duration (years)')
        ax[1].set_title('Final Duration Level by Quarter (median + IQR)')
        style_axes(ax[1], grid_axis='y')
        style_legend(ax[1], loc='upper left')

        # Enforce shared project time-axis rules: start at 1999, labels from 2000 every 2 years.
        style_time_axis(
            ax[1],
            x_min=pd.Timestamp('1999-01-01'),
            x_max=q_cov['quarter_ts'].max(),
            x_ticks=q_cov['quarter_ts'],
            date_fmt='%Y',
        )

        plt.tight_layout()
        STEP11_FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(STEP11_FIG_PATH, dpi=300, bbox_inches='tight')
        print(f'Saved Step 11 figure: {STEP11_FIG_PATH.resolve()}')
        plt.show()


## 13) Output


In [ ]:
# Save remapped full-Euro500 output with only merge keys + final output columns
if 'df_euro500_final' not in globals():
    raise ValueError('df_euro500_final not found. Run Step 11 before Step 13 output save.')

save_cols = [c for c in ['RIC', 'firm_id', 'effective_date', 'date'] if c in df_euro500_final.columns] + final_output_cols

df_to_save = df_euro500_final[save_cols].copy()

df_to_save.to_parquet(OUTPUT_PATH, index=False)

print(f'Saved (intermediate): {OUTPUT_PATH.resolve()}')
print(f'Rows saved (FULL Euro500): {len(df_to_save):,}')
print('Saved only merge keys plus final output columns.')
